# LLaVA-OneVision Baseline on MindCube
**Before running:** Upload your `data_download/data/` folder to Google Drive as `MyDrive/MindCube/data/`
so the structure is:
- `MyDrive/MindCube/data/raw/MindCube_tinybench.jsonl`
- `MyDrive/MindCube/data/other_all_image/...`

Then set Runtime → Change runtime type → **A100 GPU** (or T4).

In [1]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install -q transformers>=4.45.0 accelerate>=0.27.0 huggingface_hub pillow tqdm

In [2]:
# ── 2. Mount Google Drive ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH  = "/content/drive/MyDrive/MindCube/data/raw/MindCube_tinybench.jsonl"
IMAGE_ROOT = "/content/drive/MyDrive/MindCube/data/"

# Quick check
import pathlib
assert pathlib.Path(DATA_PATH).exists(), f"Not found: {DATA_PATH}"
print("Data found.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data found.


In [3]:
# ── 3. Dataset loader ────────────────────────────────────────────────────────
import json
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset

_HEADER = "Look at these images carefully. They show a scene from different viewpoints.\n\n"
_FOOTER = "\n\nAnswer with one letter only (A, B, C, or D)."


class MindCubeDataset(Dataset):
    def __init__(self, jsonl_path, image_root, max_samples=None):
        self.image_root = Path(image_root)
        self.samples = []
        with open(jsonl_path) as f:
            for line in f:
                line = line.strip()
                if line:
                    self.samples.append(json.loads(line))
                    if max_samples and len(self.samples) >= max_samples:
                        break

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        record = self.samples[idx]
        images = [Image.open(self.image_root / r).convert("RGB") for r in record["images"]]
        prompt = _HEADER + record["question"] + _FOOTER
        return {
            "id": record["id"],
            "images": images,
            "prompt": prompt,
            "gt_answer": record["gt_answer"],
            "setting": record["id"].split("_")[0].lower(),
        }


print("Dataset class ready.")

Dataset class ready.


In [4]:
# ── 4. Load model ────────────────────────────────────────────────────────────
import torch, gc
from transformers import LlavaOnevisionForConditionalGeneration, AutoProcessor

MODEL_ID = "/content/drive/MyDrive/models/llava-onevision-qwen2-7b-ov-hf"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print(f"Loading {MODEL_ID} on {device} ...")
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = LlavaOnevisionForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto",
    attn_implementation="sdpa",  # memory-efficient: doesn't materialise full n² attention matrix
)
model.eval()

# Disable AnyRes tiling: each image becomes 1 tile (729 tokens) instead of 8-16
processor.image_processor.do_image_splitting = False

gc.collect()
torch.cuda.empty_cache()
print(f"Model loaded. GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} GB allocated.")

The image processor of type `LlavaOnevisionImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading /content/drive/MyDrive/models/llava-onevision-qwen2-7b-ov-hf on cuda ...


Loading weights:   0%|          | 0/765 [00:00<?, ?it/s]

Model loaded. GPU memory: 16.1 GB allocated.


In [5]:
# ── 5. Inference helpers ─────────────────────────────────────────────────────
import re
from collections import defaultdict
from tqdm.notebook import tqdm

_TAG  = re.compile(r"<answer>\s*([A-E])", re.I)
_DECL = re.compile(r"(?:the\s+answer\s+is|answer\s*:)\s*([A-E])\.?", re.I)
_LINE = re.compile(r"^\s*([A-E])[\.):]?\s*$", re.I | re.M)
_ANY  = re.compile(r"([A-E])", re.I)


def extract_answer(text):
    for pat in [_TAG, _DECL]:
        m = pat.search(text)
        if m:
            return m.group(1).upper()
    for pat in [_LINE, _ANY]:
        ms = pat.findall(text)
        if ms:
            return ms[-1].upper()
    return None


@torch.inference_mode()
def generate(images, prompt, max_new_tokens=128):
    # No image resize, no system prompt — both hurt generation quality for this model
    content = [{"type": "image"} for _ in images] + [{"type": "text", "text": prompt}]
    conversation = [{"role": "user", "content": content}]
    text = processor.apply_chat_template(conversation, add_generation_prompt=True)
    inputs = processor(images=images, text=text, return_tensors="pt").to(device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated = out[0][inputs["input_ids"].shape[1]:]
    return processor.decode(generated, skip_special_tokens=True).strip()


print("Inference helpers ready.")

Inference helpers ready.


In [6]:
# ── 5b. Sanity check (3 samples) ─────────────────────────────────────────────
ds_smoke = MindCubeDataset(DATA_PATH, IMAGE_ROOT, max_samples=3)
for i in range(len(ds_smoke)):
    s = ds_smoke[i]
    raw = generate(s["images"], s["prompt"])
    pred = extract_answer(raw)
    print(f"[{s['id']}]  gt={s['gt_answer']}  pred={pred}  raw={repr(raw[:120])}")
    for img in s["images"]:
        img.close()

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


[among_group693_q1_5_2]  gt=C  pred=D  raw='D'


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


[among_group458_q0_2_3]  gt=A  pred=A  raw='A'
[among_group603_q1_1_2]  gt=B  pred=C  raw='C'


In [7]:
# ── 6. Full evaluation ───────────────────────────────────────────────────────
MAX_SAMPLES = None  # None = all 1050; set to 10 for a quick test
OUT_PATH = "/content/baseline_results.jsonl"

dataset = MindCubeDataset(DATA_PATH, IMAGE_ROOT, MAX_SAMPLES)
print(f"{len(dataset)} samples")

results = []
with open(OUT_PATH, "w") as f_out:
    for idx in tqdm(range(len(dataset))):
        sample = dataset[idx]
        try:
            raw = generate(sample["images"], sample["prompt"])
            predicted = extract_answer(raw)
            error = None
        except Exception as e:
            raw, predicted, error = "", None, str(e)
            tqdm.write(f"[WARN] {sample['id']}: {str(e)[:120]}")

        for img in sample["images"]:
            img.close()

        gt = (sample["gt_answer"] or "").upper()
        result = {
            "id": sample["id"],
            "setting": sample["setting"],
            "gt_answer": gt,
            "predicted": predicted,
            "correct": predicted is not None and predicted == gt,
            "raw_output": raw,
            **(({"error": error}) if error else {}),
        }
        results.append(result)
        f_out.write(json.dumps(result) + "\n")

        if idx % 50 == 0:
            torch.cuda.empty_cache()
            gc.collect()

print(f"Done. Results saved to {OUT_PATH}")

1050 samples


  0%|          | 0/1050 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for

Done. Results saved to /content/baseline_results.jsonl


In [8]:
# ── 7. Metrics ───────────────────────────────────────────────────────────────
overall = {"correct": 0, "total": 0}
by_setting = defaultdict(lambda: {"correct": 0, "total": 0})

for r in results:
    ok = r["predicted"] is not None and r["predicted"] == r["gt_answer"]
    overall["total"] += 1
    overall["correct"] += int(ok)
    s = r["setting"]
    by_setting[s]["total"] += 1
    by_setting[s]["correct"] += int(ok)

acc = lambda d: d["correct"] / d["total"] if d["total"] else 0.0

print(f"\n{'='*44}")
print(f"  Overall accuracy : {acc(overall):.3f}  ({overall['correct']}/{overall['total']})")
print(f"  Random baseline  : 0.250")
print(f"{'='*44}")
print(f"  {'Setting':<18}  {'Acc':>6}  Correct/Total")
print(f"  {'-'*40}")
for s, m in sorted(by_setting.items()):
    print(f"  {s:<18}  {acc(m):>6.3f}  {m['correct']}/{m['total']}")
print(f"{'='*44}")
unanswered = sum(1 for r in results if r["predicted"] is None)
errors = sum(1 for r in results if "error" in r)
print(f"Unanswered: {unanswered}/{len(results)}  (errors: {errors})")


  Overall accuracy : 0.460  (483/1050)
  Random baseline  : 0.250
  Setting                Acc  Correct/Total
  ----------------------------------------
  among                0.418  251/600
  around               0.652  163/250
  rotation             0.345  69/200
Unanswered: 0/1050  (errors: 0)


In [9]:
# ── 8. Download results ───────────────────────────────────────────────────────
from google.colab import files
files.download(OUT_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>